# QKD ViT-B/16 Kaggle Configuration

Run the repository clone cell first, then configure the CIFAR-100 dataset, optional ViT-B/16-IN21K checkpoint, output folder, and training parameters in the next cell. The notebook runs `main.py` and source code from the cloned repository; the dataset and optional checkpoint remain Kaggle inputs.

In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/your-username/your-repository.git"  # Replace with the repository URL.
REPO_ROOT = Path(os.environ.get("QKD_REPO_ROOT", "/kaggle/working/QG_repro"))

if REPO_ROOT.exists():
    if not (REPO_ROOT / ".git").is_dir():
        raise FileExistsError(f"{REPO_ROOT} exists but is not a Git repository.")
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    print(f"Updated repository: {REPO_ROOT}")
else:
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(REPO_ROOT)], check=True)
    print(f"Cloned repository: {REPO_ROOT}")

os.environ["QKD_REPO_ROOT"] = str(REPO_ROOT)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get('QKD_REPO_ROOT', '/kaggle/working/QG_repro'))
DATA_ROOT = Path(os.environ.get('QKD_CIFAR100_ROOT', '/kaggle/input/cifar_100'))
CHECKPOINT_PATH = Path(
    os.environ.get(
        'QKD_VIT_B16_IN21K_WEIGHTS',
        '/kaggle/input/vit-b16-in21k/checkpoint.pth',
    )
)
OUTPUT_ROOT = Path(os.environ.get('QKD_OUTPUT_ROOT', '/kaggle/working/qkd_outputs'))

CONFIG = {
    'seed': 1993,
    'batch_size': 32,
    'num_workers': 2,
    'epochs_per_task': 20,
    'learning_rate': 0.05,
    'bottleneck_dim': 64,
    'initial_classes': 0,
    'incremental_classes': 10,
}

main_file = REPO_ROOT / 'main.py'
if not main_file.is_file():
    raise FileNotFoundError(f'Cannot find {main_file}; run the clone cell first.')

splits_module = REPO_ROOT / 'src' / 'qkd' / 'data' / 'splits.py'
if not splits_module.is_file():
    raise FileNotFoundError(
        f'Incomplete or outdated repository checkout at {REPO_ROOT}: '
        f'cannot find {splits_module}. Refresh this checkout or point QKD_REPO_ROOT '
        'to a checkout containing src/qkd/data/splits.py.'
    )
print(f'Using repository: {REPO_ROOT.resolve()}')
print(f'QKD data module: {splits_module.resolve()}')

command = [
    sys.executable, str(main_file),
    '--data-root', str(DATA_ROOT),
    '--checkpoint-path', str(CHECKPOINT_PATH),
    '--output-dir', str(OUTPUT_ROOT),
    '--seed', str(CONFIG['seed']),
    '--batch-size', str(CONFIG['batch_size']),
    '--num-workers', str(CONFIG['num_workers']),
    '--epochs-per-task', str(CONFIG['epochs_per_task']),
    '--learning-rate', str(CONFIG['learning_rate']),
    '--bottleneck-dim', str(CONFIG['bottleneck_dim']),
    '--initial-classes', str(CONFIG['initial_classes']),
    '--incremental-classes', str(CONFIG['incremental_classes']),
]
subprocess.run(command, cwd=REPO_ROOT, check=True)